In [0]:
# Verifique o nome exato das colunas na Bronze
spark.table("mvp.staging.bronze_diabetes_raw").printSchema()

In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"
TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"
TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"

df_bronze = spark.table(TABELA_BRONZE)

# 1. Padronização de nomes de colunas
for c in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

# 2. Conversão segura tratando explicitamente os valores textuais
df_silver = df_bronze \
    .withColumn("age", F.col("age").cast("int")) \
    .withColumn("bmi", F.col("bmi").cast("double")) \
    .withColumn("fasting_blood_sugar", F.col("fasting_blood_sugar").cast("double")) \
    .withColumn(
        "family_history_diabetes",
        F.when(F.lower(F.trim(F.col("family_history_diabetes"))).isin(["yes", "true", "1"]), 1)
        .otherwise(0)
    ) \
    .withColumn(
        "diabetes",
        # Trata valores 'high', 'yes', 'true', '1' ou numéricos convertidos previamente via try_cast
        F.when(F.lower(F.trim(F.col("diabetes_risk"))).isin(["high", "yes", "true", "1"]), 1)
        .when(F.expr("try_cast(diabetes_risk as int)") >= 1, 1)
        .otherwise(0)
    ) \
    .withColumn(
        "physical_activity_level", 
        F.lower(F.trim(F.col("physical_activity_level")))
    )

# 3. Seleção dos atributos essenciais
colunas_finais = ["age", "bmi", "fasting_blood_sugar", "family_history_diabetes", "physical_activity_level", "diabetes"]
df_silver = df_silver.select(*colunas_finais)

# 4. Deduplicação e remoção de nulos
df_silver = df_silver.dropDuplicates() \
    .filter(F.col("age").isNotNull() & F.col("bmi").isNotNull() & F.col("fasting_blood_sugar").isNotNull())

# 5. Gravando a tabela Silver
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_SILVER)

print("Tabela Silver reprocessada com sucesso sem erros de cast!")

In [0]:
%sql
-- Verificar se restou algum registro nulo ou duplicado
SELECT 
  COUNT(*) AS total_registros,
  SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS nulos_idade,
  SUM(CASE WHEN bmi IS NULL THEN 1 ELSE 0 END) AS nulos_imc
FROM mvp.staging.silver_diabetes_clean;

In [0]:
# Substitui NULL por 'not_informed' (não informado) na Silver
df_silver = df_silver.withColumn(
    "physical_activity_level",
    F.when(F.col("physical_activity_level").isNull(), "not_informed")
    .otherwise(F.lower(F.trim(F.col("physical_activity_level"))))
)

In [0]:
# Filtra Nulos reais e também a palavra 'null' em texto
df_silver = df_silver.dropDuplicates() \
    .filter(
        F.col("age").isNotNull() & 
        F.col("bmi").isNotNull() & 
        F.col("fasting_blood_sugar").isNotNull() &
        F.col("physical_activity_level").isNotNull() &
        (F.lower(F.col("physical_activity_level")) != "null")
    )